# 01 — TMDB Poster Scraping
Hedef: 10.000–12.000 film afişi + `labels.csv`

In [1]:
# --- Google Colab: Drive bağlantısı ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT  = '/content/drive/MyDrive/film-genre-project'   # kod (git repo)
# DATA_ROOT  = '/content/drive/MyDrive/film-genre-project-data'  # veri

# --- Lokalde çalıştırma ---
from pathlib import Path
CODE_ROOT = Path('..').resolve()
DATA_ROOT = CODE_ROOT  # lokalde proje klasörünün kendisi

POSTERS_DIR  = DATA_ROOT / 'posters'
LABELS_PATH  = DATA_ROOT / 'labels.csv'

POSTERS_DIR.mkdir(exist_ok=True)
print('Kod:', CODE_ROOT)
print('Veri:', DATA_ROOT)

Kod: C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project
Veri: C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project


In [2]:
import subprocess

SCRAPY_DIR = CODE_ROOT / 'src' / 'scraper'
SCRAPY_BIN = CODE_ROOT / '.venv' / 'Scripts' / 'scrapy.exe'

# Colab'da:
# !pip install scrapy requests pillow -q
# SCRAPY_BIN = Path('scrapy')  # PATH'ten

print('Scrapy dir:', SCRAPY_DIR)
print('Scrapy bin:', SCRAPY_BIN)

Scrapy dir: C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project\src\scraper
Scrapy bin: C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project\.venv\Scripts\scrapy.exe


---

## ID Tabanlı Tam Çekim (Önerilen)

Listing spider (`tmdb`) 429 ban'ına ~600–700 filmde çarpıyor çünkü sıralı listing sayfaları rate-limit'i tetikliyor.

**Bu yaklaşım:** TMDB'nin günlük halka açık export'undan ~600k film ID'si indirilir, doğrudan `/movie/{id}` detay sayfaları ziyaret edilir. Listing sayfası yok → listing ban'ı yok.

İki adım:
1. `tmdb_movie_ids.txt` oluştur (aşağıdaki hücre)
2. `tmdb_id` spider'ını başlat (bir sonraki hücre)

In [3]:
import gzip, json, datetime, requests
from io import BytesIO
import pandas as pd

# TMDB günlük export — bugün yoksa dünü dene (export bazen 1 gün gecikmeli)
for delta in range(3):
    date_str = (datetime.date.today() - datetime.timedelta(days=delta)).strftime("%m_%d_%Y")
    export_url = f"https://files.tmdb.org/p/exports/movie_ids_{date_str}.json.gz"
    resp = requests.get(export_url, timeout=60)
    if resp.status_code == 200:
        print(f"Export indirildi: {export_url}  ({len(resp.content)/1e6:.1f} MB)")
        break
else:
    raise RuntimeError("TMDB export son 3 günde bulunamadı")

# ID'leri parse et (adult filmleri hariç tut)
all_ids = []
with gzip.open(BytesIO(resp.content)) as f:
    for line in f:
        obj = json.loads(line)
        if not obj.get("adult", False):
            all_ids.append(str(obj["id"]))
print(f"Toplam ID   : {len(all_ids):,}")

# Zaten çekilmiş ID'leri çıkar
seen_ids = set()
if LABELS_PATH.exists() and LABELS_PATH.stat().st_size > 0:
    seen_ids = set(pd.read_csv(LABELS_PATH)["tmdb_id"].astype(str))
    print(f"Zaten çekilmiş: {len(seen_ids):,}  |  Atlanacak")

remaining_ids = [i for i in all_ids if i not in seen_ids]
print(f"Çekilecek   : {len(remaining_ids):,}")

IDS_FILE = DATA_ROOT / "tmdb_movie_ids.txt"
IDS_FILE.write_text("\n".join(remaining_ids), encoding="utf-8")
print(f"Kaydedildi  : {IDS_FILE}")

Export indirildi: https://files.tmdb.org/p/exports/movie_ids_05_13_2026.json.gz  (25.9 MB)
Toplam ID   : 1,195,029
Zaten çekilmiş: 11,222  |  Atlanacak
Çekilecek   : 1,183,809
Kaydedildi  : C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project\tmdb_movie_ids.txt


In [ ]:
import subprocess, time, datetime, pandas as pd
from IPython.display import clear_output

LOG_PATH      = DATA_ROOT / 'scrapy_id.log'
IDS_FILE      = DATA_ROOT / 'tmdb_movie_ids.txt'
TARGET        = 12_000
POLL_INTERVAL = 15

def _count():
    try:
        if LABELS_PATH.exists() and LABELS_PATH.stat().st_size > 0:
            return len(pd.read_csv(LABELS_PATH))
    except Exception:
        pass
    return 0

proc = subprocess.Popen(
    [str(SCRAPY_BIN), 'crawl', 'tmdb_id',
     '-a', f'ids_file={IDS_FILE}',       # -a: spider arg, -s: setting — ikisi farklı!
     '-s', f'IMAGES_STORE={POSTERS_DIR}',
     '-s', f'LABELS_PATH={LABELS_PATH}',
     '-L', 'WARNING'],
    cwd=str(SCRAPY_DIR),
    stdout=open(LOG_PATH, 'a', encoding='utf-8'),
    stderr=subprocess.STDOUT,
)
print(f"tmdb_id spider başlatıldı — PID: {proc.pid}")

start_time  = time.time()
start_count = _count()
recent      = []

try:
    while proc.poll() is None:
        now     = time.time()
        current = _count()
        elapsed = now - start_time

        recent.append((now, current))
        recent = [(t, c) for t, c in recent if now - t <= 300]

        if len(recent) >= 2:
            dt, dc   = recent[-1][0] - recent[0][0], recent[-1][1] - recent[0][1]
            speed_hr = dc / dt * 3600 if dt > 0 else 0
        else:
            speed_hr = 0

        remaining = max(TARGET - current, 0)
        if speed_hr > 0:
            eta_dt   = datetime.datetime.now() + datetime.timedelta(hours=remaining / speed_hr)
            eta_str  = eta_dt.strftime('%d.%m.%Y %H:%M')
            eta_left = str(datetime.timedelta(seconds=int(remaining / speed_hr * 3600)))
        else:
            eta_str = eta_left = 'hesaplanıyor...'

        pct = min(current / TARGET, 1.0)
        bar = '█' * int(40 * pct) + '░' * (40 - int(40 * pct))

        clear_output(wait=True)
        print("TMDB ID Spider — Canlı İzleme  [listing ban yok]")
        print("─" * 54)
        print(f"  [{bar}] {pct*100:.1f}%")
        print(f"  Çekilen (toplam)    : {current:>6,} / {TARGET:,}")
        print(f"  Bu oturumda         : +{current - start_count:,}")
        print(f"  İndirilen poster    : {len(list(POSTERS_DIR.glob('*.jpg'))):,}")
        print("─" * 54)
        print(f"  Hız                 : {speed_hr:>6.0f} film/saat")
        print(f"  Kalan               : {remaining:,} film")
        print(f"  Kalan süre (tahmini): {eta_left}")
        print(f"  Tahmini bitiş       : {eta_str}")
        print("─" * 54)
        print(f"  Geçen süre          : {str(datetime.timedelta(seconds=int(elapsed)))}")
        print(f"  PID: {proc.pid}  |  Log: {LOG_PATH}")
        print()
        print("  Hücreyi durdurursan Scrapy da durur.")
        print("  Tekrar başlatmadan önce üstteki ID hücresini tekrar çalıştır (seen_ids güncellenir).")

        if current >= TARGET:
            print(f"\n  Hedef {TARGET:,} filme ulaşıldı!")
            break

        time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    proc.terminate()
    print(f"\nDurduruldu — {_count():,} film çekildi.")

proc.wait()
print(f"\nTamamlandı. Toplam: {_count():,} film")

TMDB ID Spider — Canlı İzleme  [listing ban yok]
──────────────────────────────────────────────────────
  [████████████████████████████████████████] 100.0%
  Çekilen (toplam)    : 12,001 / 12,000
  Bu oturumda         : +779
  İndirilen poster    : 11,973
──────────────────────────────────────────────────────
  Hız                 :   1710 film/saat
  Kalan               : 0 film
  Kalan süre (tahmini): 0:00:00
  Tahmini bitiş       : 14.05.2026 01:48
──────────────────────────────────────────────────────
  Geçen süre          : 0:35:55
  PID: 62300  |  Log: C:\Users\Kerem\Desktop\MakinÖğrenmesi\film-genre-project\scrapy_id.log

  Hücreyi durdurursan Scrapy da durur.
  Tekrar başlatmadan önce üstteki ID hücresini tekrar çalıştır (seen_ids güncellenir).

  Hedef 12,000 filme ulaşıldı!
